# EDA - Exploratórna Analýza Dát

Tento notebook obsahuje detailný prieskum dát potrebný na pochopenie:
- Štruktúry a kvality dát
- Rozdelenia fraud prípadov
- Vzťahov medzi premennými
- Anomálií a chýb v dátach

Na základe zistení sa nastaví proces data preprocessingu.

In [2]:
import pandas as pd
import numpy as np
from typing import List, Optional
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
from pathlib import Path
import pyarrow.parquet as pq
import warnings
warnings.filterwarnings('ignore')

# Nastavenie štýlu grafov
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

In [3]:
# =============================================================================
# 1. NAČÍTANIE DÁTOVÝCH ZDROJOV
# =============================================================================
# Načítavame tri hlavné tabuľky:
# - cards: informácie o kartách
# - transactions: jednotlivé transakcie
# - users: informácie o používateľoch

root = Path.cwd().parent
data_dir = root / "data" / "raw"

cards = pq.read_table(data_dir / "cards.pq").to_pandas()
transactions = pq.read_table(data_dir / "transactions.pq").to_pandas()
users = pq.read_table(data_dir / "users.pq").to_pandas()

## Fáza 1: Základné Informácie o Vstupných Dátach

In [4]:
# =============================================================================
# 2. DÁTOVÉ TYPY A PRVÝ POHĽAD NA DÁTA
# =============================================================================

print("DÁTOVÉ TYPY - TRANSAKCIE\n")
print(transactions.dtypes)

print("\n\n--- Prvých 5 transakcií ---")
display(transactions.head())

print("\n\nDÁTOVÉ TYPY - POUŽÍVATELIA\n")
print(users.dtypes)

print("\n\n--- Prvých 5 používateľov ---")
display(users.head())

print("\n\nDÁTOVÉ TYPY - KARTY\n")
print(cards.dtypes)

print("\n\n--- Prvých 5 kariet ---")
display(cards.head())

DÁTOVÉ TYPY - TRANSAKCIE

User                int64
Card                int64
Year                int64
Month               int64
Day                 int64
Time                  str
Amount                str
Use Chip              str
Merchant Name       int64
Merchant City         str
Merchant State        str
Zip               float64
MCC                 int64
Errors?               str
Is Fraud?             str
dtype: object


--- Prvých 5 transakcií ---


,User,Card,Year,Month,Day,Time,Amount,Use Chip,Merchant Name,Merchant City,Merchant State,Zip,MCC,Errors?,Is Fraud?
0,0,0,2018,1,2,06:28,$130.95,Chip Transaction,5817218446178736267,La Verne,CA,91750.0,5912,NaN,No
1,0,0,2018,1,5,06:02,$129.34,Chip Transaction,-727612092139916043,Monterey Park,CA,91754.0,5411,NaN,No
2,0,0,2018,1,5,17:56,$8.54,Online Transaction,6455213054093379528,ONLINE,NaN,NaN,5815,NaN,No
3,0,0,2018,1,6,06:23,$115.71,Chip Transaction,4722913068560264812,Alhambra,CA,91801.0,5411,NaN,No
4,0,0,2018,1,11,06:02,$107.50,Chip Transaction,2027553650310142703,Mira Loma,CA,91752.0,5541,NaN,No




DÁTOVÉ TYPY - POUŽÍVATELIA

User                             int64
Person                             str
Current Age                      int64
Retirement Age                   int64
Birth Year                       int64
Birth Month                      int64
Gender                             str
Address                            str
Apartment                      float64
City                               str
State                              str
Zipcode                          int64
Latitude                       float64
Longitude                      float64
Per Capita Income - Zipcode        str
Yearly Income - Person             str
Total Debt                         str
FICO Score                       int64
Num Credit Cards                 int64
dtype: object


--- Prvých 5 používateľov ---


,User,Person,Current Age,Retirement Age,Birth Year,Birth Month,Gender,Address,Apartment,City,State,Zipcode,Latitude,Longitude,Per Capita Income - Zipcode,Yearly Income - Person,Total Debt,FICO Score,Num Credit Cards
0,0,Hazel Robinson,53,66,1966,11,Female,462 Rose Lane,NaN,La Verne,CA,91750,34.15,-117.76,$29278,$59696,$127613,787,5
1,1,Sasha Sadr,53,68,1966,12,Female,3606 Federal Boulevard,NaN,Little Neck,NY,11363,40.76,-73.74,$37891,$77254,$191349,701,5
2,2,Saanvi Lee,81,67,1938,11,Female,766 Third Drive,NaN,West Covina,CA,91792,34.02,-117.89,$22681,$33483,$196,698,5
3,3,Everlee Clark,63,63,1957,1,Female,3 Madison Street,NaN,New York,NY,10069,40.71,-73.99,$163145,$249925,$202328,722,4
4,4,Kyle Peterson,43,70,1976,9,Male,9620 Valley Stream Drive,NaN,San Francisco,CA,94117,37.76,-122.44,$53797,$109687,$183855,675,1




DÁTOVÉ TYPY - KARTY

User                     int64
CARD INDEX               int64
Card Brand                 str
Card Type                  str
Card Number              int64
Expires                    str
CVV                      int64
Has Chip                   str
Cards Issued             int64
Credit Limit               str
Acct Open Date             str
Year PIN last Changed    int64
Card on Dark Web           str
dtype: object


--- Prvých 5 kariet ---


,User,CARD INDEX,Card Brand,Card Type,Card Number,Expires,CVV,Has Chip,Cards Issued,Credit Limit,Acct Open Date,Year PIN last Changed,Card on Dark Web
0,0,0,Visa,Debit,4344676511950444,12/2022,623,YES,2,$24295,09/2002,2008,No
1,0,1,Visa,Debit,4956965974959986,12/2020,393,YES,2,$21968,04/2014,2014,No
2,0,2,Visa,Debit,4582313478255491,02/2024,719,YES,2,$46414,07/2003,2004,No
3,0,3,Visa,Credit,4879494103069057,08/2024,693,NO,1,$12400,01/2003,2012,No
4,0,4,Mastercard,Debit (Prepaid),5722874738736011,03/2009,75,YES,1,$28,09/2008,2009,No


In [ ]:

# =============================================================================
# 3. ROZMERY A ŠTRUKTÚRA DATASETOV
# =============================================================================

print("ROZMERY DATASETOV")
print("=" * 60)
print(f"Transakcie:    {transactions.shape[0]:>10} riadkov, {transactions.shape[1]:>3} stĺpcov")
print(f"Používatelia:  {users.shape[0]:>10} riadkov, {users.shape[1]:>3} stĺpcov")
print(f"Karty:         {cards.shape[0]:>10} riadkov, {cards.shape[1]:>3} stĺpcov")
print("=" * 60)



ROZMERY DATASETOV
Transakcie:       3782053 riadkov,  15 stĺpcov
Používatelia:        2000 riadkov,  19 stĺpcov
Karty:               6146 riadkov,  13 stĺpcov


## Fáza 2: Spájanie Dát

In [5]:
# Spojíme vstupné tabuľky do jedného datasetu, ktorý bude obsahovat informácie na úrovni transakcií.

# Kontrola kľúčových polí
print("\n--- Unikátnosť kľúčových polí ---")
print(f"Unikátni používatelia:    {users['User'].nunique():>5} z {len(users):>5}")
print(f"Unikátne karty (index):   {cards['CARD INDEX'].nunique():>5} z {len(cards):>5}")
print(f"Unikátne karty v transc.: {transactions['Card'].nunique():>5} z {len(transactions):>5}")

#  UPOZORNENIE: Je len 9 unikátnych CARD INDEX! To znamená, že tieto 9 hodnôt sa opakujú pre rôznych používateľov. Pri merge bude potrebné použiť kombináciu 'CARD INDEX' a 'User' ako kľúč, nie len 'CARD INDEX'.

print("\nSPÁJANIE TABULIEK...")
print("Spájame: Transakcie + Používatelia + Karty")

data = (
    transactions
    .merge(users, on='User', how='left')
    .merge(cards,
           left_on=['User', 'Card'],
           right_on=['User', 'CARD INDEX'],
           how='left')
)

print(f"Spájanie hotové!")
print(f"   Výsledný dataset: {data.shape[0]} riadkov, {data.shape[1]} stĺpcov")

print("\n\nDÁTOVÉ TYPY stĺpcov v zjednotenom datasete\n")
print(data.dtypes)

print("\nPrvých 5 riadkov zjednotených dát:")
display(data.head())



--- Unikátnosť kľúčových polí ---
Unikátni používatelia:     2000 z  2000
Unikátne karty (index):       9 z  6146
Unikátne karty v transc.:     9 z 3782053

SPÁJANIE TABULIEK...
Spájame: Transakcie + Používatelia + Karty
Spájanie hotové!
   Výsledný dataset: 3782053 riadkov, 45 stĺpcov


DÁTOVÉ TYPY stĺpcov v zjednotenom datasete

User                             int64
Card                             int64
Year                             int64
Month                            int64
Day                              int64
Time                               str
Amount                             str
Use Chip                           str
Merchant Name                    int64
Merchant City                      str
Merchant State                     str
Zip                            float64
MCC                              int64
Errors?                            str
Is Fraud?                          str
Person                             str
Current Age                      int64
Ret

,User,Card,Year,Month,Day,Time,Amount,Use Chip,Merchant Name,Merchant City,...,Card Type,Card Number,Expires,CVV,Has Chip,Cards Issued,Credit Limit,Acct Open Date,Year PIN last Changed,Card on Dark Web
0,0,0,2018,1,2,06:28,$130.95,Chip Transaction,5817218446178736267,La Verne,...,Debit,4344676511950444,12/2022,623,YES,2,$24295,09/2002,2008,No
1,0,0,2018,1,5,06:02,$129.34,Chip Transaction,-727612092139916043,Monterey Park,...,Debit,4344676511950444,12/2022,623,YES,2,$24295,09/2002,2008,No
2,0,0,2018,1,5,17:56,$8.54,Online Transaction,6455213054093379528,ONLINE,...,Debit,4344676511950444,12/2022,623,YES,2,$24295,09/2002,2008,No
3,0,0,2018,1,6,06:23,$115.71,Chip Transaction,4722913068560264812,Alhambra,...,Debit,4344676511950444,12/2022,623,YES,2,$24295,09/2002,2008,No
4,0,0,2018,1,11,06:02,$107.50,Chip Transaction,2027553650310142703,Mira Loma,...,Debit,4344676511950444,12/2022,623,YES,2,$24295,09/2002,2008,No


## Fáza 3: Oboznámenie s Dátami

In [1]:
print("ČASOVÝ ROZSAH DÁT:")
print("=" * 80)

# Vytvoríme dátum zo stĺpcov Year / Month / Day, pretože priamy stĺpec Date v datasetu neexistuje.
transactions['Date'] = pd.to_datetime(
    transactions[['Year', 'Month', 'Day']].rename(columns={'Year': 'year', 'Month': 'month', 'Day': 'day'}),
    errors='coerce'
)

# Stlpec transactions['Amount'] obsahuje hodnoty vo formátu "$123.45". Odstránime znak dolára a prevedieme na float pre ďalšie analýzy.
transactions['Amount'] = transactions['Amount'].str.replace('$','').astype(float)

print(f"\nCasovy rozsah transakcií: {transactions['Date'].min()} - {transactions['Date'].max()}")


ČASOVÝ ROZSAH DÁT:


NameError: name 'pd' is not defined

## Fáza 2: Kontrola Kvality Dát

In [26]:
# =============================================================================
# 4. CHÝBAJÚCE HODNOTY
# =============================================================================

print("CHÝBAJÚCE HODNOTY (NULL)")
print("=" * 80)

missing = data.isnull().sum()
if missing.sum() == 0:
    print("  Žiadne chýbajúce hodnoty")
else:
    for col in missing[missing > 0].index:
        pct = (missing[col] / len(data)) * 100
        print(f"  {col:30s}: {missing[col]:>7} ({pct:>5.2f}%)")

CHÝBAJÚCE HODNOTY (NULL)
  Merchant State                :  473483 (12.52%)
  Zip                           :  500588 (13.24%)
  Errors?                       : 3722382 (98.42%)
  Apartment                     : 2753083 (72.79%)


In [ ]:
# =============================================================================
# 5. DUPLICITNÉ ZÁZNAMY
# =============================================================================

print(" DUPLICITNÉ ZÁZNAMY")
print("=" * 80)

# Kontrola úplne duplicitných riadkov
dup = data.duplicated().sum()

print(f"Transakcie - úplne duplikáty:  {dup} ({(dup/len(data))*100:.5f}%)")



 DUPLICITNÉ ZÁZNAMY
Transakcie - úplne duplikáty:  6 (0.00016%)


## Fáza 3: Analýza TARGET Premennej (Fraud)

In [30]:
# =============================================================================
# 6. DISTRIBÚCIA FRAÚD PRÍPADOV
# =============================================================================

print("ANALÝZA FRAUD PRÍPADOV")
print("=" * 80)

fraud_counts = transactions['Is Fraud?'].value_counts(dropna=False)
fraud_pcts = (transactions['Is Fraud?'].value_counts(normalize=True, dropna=False) * 100)

fraud_df = pd.DataFrame({
    'Počet': fraud_counts,
    'Percento': fraud_pcts.round(2)
})

print("\nRozdelenie fraud/non-fraud transakcií:")
print(fraud_df)

print("\nVAROVANIE: Dáta sú VYSOKO NEVYVÁŽENÉ (imbalanced)!")
print("    -> Pri modelovaní budeme musieť použiť metódy pre prácu s nevyváženými dátami")

ANALÝZA FRAUD PRÍPADOV

Rozdelenie fraud/non-fraud transakcií:
             Počet  Percento
Is Fraud?                   
No         3777475     99.88
Yes           4578      0.12

VAROVANIE: Dáta sú VYSOKO NEVYVÁŽENÉ (imbalanced)!
    -> Pri modelovaní budeme musieť použiť metódy pre prácu s nevyváženými dátami


## Fáza 4: Analýza Kategorických Premenných

In [21]:
# =============================================================================
# 7. ANALÝZA KATEGORICKÝCH PREMENNÝCH V TRANSAKCIÁCH
# =============================================================================

print("KATEGORICKÉ PREMENNÉ V TRANSAKCIÁCH")
print("=" * 80)

categorical_cols = ['Type', 'Errors?', 'MCC', 'Merchant State', 'Merchant City']

for col in categorical_cols:
    if col in transactions.columns:
        print(f"\n{col}: {transactions[col].nunique()} unikátnych hodnôt")
        print("Top 5:")
        print(transactions[col].value_counts().head(5))
        print()

# Analýza vzťahu medzi Type a Fraud
print("\n" + "=" * 80)
print("VZŤAH: Transaction Type vs Fraud")
print("=" * 80)
type_fraud = pd.crosstab(transactions['Type'], transactions['Is Fraud?'], margins=True)
print(type_fraud)

fraud_rate_by_type = pd.crosstab(transactions['Type'], transactions['Is Fraud?'], normalize='index') * 100
print("\nFraud rate (%) podľa typu transakcie:")
print(fraud_rate_by_type.round(2))

KATEGORICKÉ PREMENNÉ V TRANSAKCIÁCH

Errors?: 21 unikátnych hodnôt
Top 5:
Errors?
Insufficient Balance    36506
Bad PIN                  9024
Technical Glitch         7586
Bad Card Number          2315
Bad Expiration           1847
Name: count, dtype: int64


MCC: 109 unikátnych hodnôt
Top 5:
MCC
5411    450761
5499    406039
5541    395039
5812    281374
5912    218234
Name: count, dtype: int64


Merchant State: 152 unikátnych hodnôt
Top 5:
Merchant State
CA    399135
TX    273732
NY    213744
FL    203698
OH    138779
Name: count, dtype: int64


Merchant City: 9833 unikátnych hodnôt
Top 5:
Merchant City
ONLINE         473483
Houston         38023
Los Angeles     27660
Miami           26345
Brooklyn        23300
Name: count, dtype: int64


VZŤAH: Transaction Type vs Fraud


KeyError: 'Type'

In [31]:
# =============================================================================
# 8. ANALÝZA NUMERICKÝCH PREMENNÝCH
# =============================================================================

print("NUMERICKÉ PREMENNÉ - ŠTATISTIKA")
print("=" * 80)

numeric_cols = data.select_dtypes(include=[np.number]).columns

print("\nZÁKLADNÉ ŠTATISTIKY:")
print(data[numeric_cols].describe().to_string())

# Analýza Amount vzťahu s Fraud
print("\n\nANALÝZA: Amount vs Fraud")
print("=" * 80)
for fraud_status in ['No', 'Yes']:
    amounts = data[data['Is Fraud?'] == fraud_status]['Amount']
    print(f"\n{fraud_status} - {len(amounts)} transakcií:")
    print(f"  Priemer: ${amounts.mean():.2f}")
    print(f"  Medián:  ${amounts.median():.2f}")
    print(f"  Min:     ${amounts.min():.2f}")
    print(f"  Max:     ${amounts.max():.2f}")
    print(f"  Std:     ${amounts.std():.2f}")

# Vizualizácia Amount distribúcie
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bez fraúdu
axes[0].hist(data[data['Is Fraud?'] == 'No']['Amount'], bins=50, color='#2ecc71', alpha=0.7, edgecolor='black')
axes[0].set_title('Amount - Normálne Transakcie', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Suma ($)')
axes[0].set_ylabel('Počet')

# S fraúdom
axes[1].hist(data[data['Is Fraud?'] == 'Yes']['Amount'], bins=50, color='#e74c3c', alpha=0.7, edgecolor='black')
axes[1].set_title('Amount - Fraúdne Transakcie', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Suma ($)')
axes[1].set_ylabel('Počet')

plt.tight_layout()
plt.show()

NUMERICKÉ PREMENNÉ - ŠTATISTIKA

ZÁKLADNÉ ŠTATISTIKY:
               User          Card          Year         Month           Day  Merchant Name           Zip           MCC   Current Age  Retirement Age    Birth Year   Birth Month     Apartment       Zipcode      Latitude     Longitude    FICO Score  Num Credit Cards    CARD INDEX   Card Number           CVV  Cards Issued  Year PIN last Changed
count  3.782053e+06  3.782053e+06  3.782053e+06  3.782053e+06  3.782053e+06   3.782053e+06  3.281465e+06  3.782053e+06  3.782053e+06    3.782053e+06  3.782053e+06  3.782053e+06  1.028970e+06  3.782053e+06  3.782053e+06  3.782053e+06  3.782053e+06      3.782053e+06  3.782053e+06  3.782053e+06  3.782053e+06  3.782053e+06           3.782053e+06
mean   1.004821e+03  1.265310e+00  2.018634e+03  6.079151e+00  1.567852e+01  -4.941558e+17  5.096764e+04  5.558671e+03  5.161998e+01    6.637183e+01  1.967562e+03  6.540997e+00  6.694271e+02  5.012971e+04  3.740527e+01 -9.137384e+01  7.114628e+02      3.5463

TypeError: Cannot perform reduction 'mean' with string dtype

## Ďalšia analýza

In [ ]:
# Možno prepokladať, že sa viacero útokov odohráva v určitých časových intervaloch (napr. večer, víkend). Preto by sme mohli analyzovat časové vzorce v dátach.
# Vykreslíme časové rozloženie fraud transakcií vzhľadom na deň v týždni.

data

In [ ]:
# Vytvoríme dátum zo stĺpcov Year / Month / Day, pretože priamy stĺpec Date v datasetu neexistuje.
transactions['Date'] = pd.to_datetime(
    transactions[['Year', 'Month', 'Day']].rename(columns={'Year': 'year', 'Month': 'month', 'Day': 'day'}),
    errors='coerce'
)

# Stlpec transactions['Amount'] obsahuje hodnoty vo formátu "$123.45". Odstránime znak dolára a prevedieme na float pre ďalšie analýzy.
transactions['Amount'] = transactions['Amount'].str.replace('$','').astype(float)

print(f"\nCasovy rozsah transakcií: {transactions['Date'].min()} - {transactions['Date'].max()}")
print(f"Suma transakcií: ${transactions['Amount'].min():.2f} - ${transactions['Amount'].max():.2f}")

## Fáza 6: Zhrnutie pre preprocessing

In [ ]:
# =============================================================================
# 10. ZHRNUTIE ZISTENÍ Z EDA
# =============================================================================

print("\n" + "=" * 80)
print("ZHRNUTIE EXPLORATÓRNEJ ANALÝZY")
print("=" * 80)

print("""
KVALITA DÁT:
   - Veľa chýbajúcich hodnot pre  (NULL)
   - Žiadne úplne duplicitné riadky


PROBLÉM - NEVYVÁŽENÉ DÁTA (CLASS IMBALANCE):
   - Fraud: {:.2f}% z celkových transakcií
   - Non-fraud: {:.2f}% z celkových transakcií
   -> Budeme musieť použiť metódy pre prácu s nevyváženými dátami 

KLÚČOVÉ ZISTENIA:
   1. Amount má rozdielnu distribúciu pre fraud vs non-fraud
   2. Niektoré typy transakcií majú vyššiu fraud rate
   3. Geografické premenné (Merchant State, City) majú potenciál
   4. FICO Score a Yearly Income sú dôležité pre profiling

NASLEDUJÚCE KROKY V PREPROCESSING:


   2. Kódovanie kategorických premenných (Gender) pomocou one-hot encoding
      Všetky yes/no stlpce (,Has Chip,Card on Dark Web) kódovat ako 1/0
      


   3. Tvorba feature engineering premenných:
      Konverzie: 
      - Time (zo str na time)
      - Amount (zo str na float)

      

      Vytvoríme nové premenné, ktoré môžu pomôcť modelu:
      - Year/Month/Day -> Date
      - Transaction Day of Week (z Date)
      - Transaction Hour (z Time)
      - Amount log-transform (pre zníženie skewness)
      - Amount diff od priemeru pre daného používateľa za posledných 30 dní (train test split až po tom, aby nedošlo k resetovaniu historie pri napočítaní priemeru)
      - First time transaction for user to this merchant (boolean)
      - Time between current transaction and last transaction for this user 
      - Merchant fraud rate (target encoding pre Merchant name, aplikovať Laplace smoothing, aby sa merchanti s malým počtom transakcií nemali extrémne hodnoty)

      
      Dropnutie irrelevantných stĺpcov:
      - CVV
      - Card Number
      - Apartment
      - Time (akonáhle sme vytvorili Transaction Hour, už tento stlpec nepotrebujeme)
      - Merchant name (ak použijeme target encoding, tento stĺpec už nepotrebujeme)
      - Person


      Normalizácia numerických premenných nie je potrebná, pretože budeme používať algoritmua, ktoré nie sú citlivý na mierku (XGBoost)
      FICO Score nebudeme transformovať do kategórií, o "kategorizáciu" sa postará XGBoost


   4. Balancovanie trénovacieho datasetu
   5. Rozdelenie na train/test sady

DÁTA PRE MODEL:
   - Počet riadkov: {rows}
   - Počet stĺpcov: {cols}
   - Cieľová premenná: Is Fraud? (Yes/No)
""".format(
    (fraud_counts.get('Yes', 0) / len(transactions) * 100),
    (fraud_counts.get('No', 0) / len(transactions) * 100),
    rows=transactions_full.shape[0],
    cols=transactions_full.shape[1]
))

print("=" * 80)
